> **📝 내 메모** — (여기에 적기. 원본 셀은 건드리지 말고 새 셀로만 추가.)

> **⚙️ 원본에 없는 셀 (배윤진 추가)** — repo 루트의 `.env`(gitignored)에서 `QISKIT_IBM_*`
> 환경변수를 읽어온다. 이 노트북은 대부분 로컬(`AerSimulator`)이고, 클라우드 셀은 `try/except`로
> 감싸져 있어 계정이 없어도 그냥 넘어간다. 커널은 `Python (qiskit-sg24)`.
>
> **이 파일은 한글 번역본.** 원본은 `1_Retrieve_Experiment_Results.ipynb`. 코드는 동일하고
> 주석·출력 문자열·설명만 번역했다. 연습문제에는 오답 해설을 추가했다.

In [1]:
import os
from pathlib import Path

env_path = next((p / ".env" for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists()), None)
if env_path:
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ[k.strip()] = v.strip()
print(f".env: {env_path or '없음'} · QISKIT_IBM_TOKEN {'설정됨' if os.getenv('QISKIT_IBM_TOKEN') else '없음'}")

.env: /home/annie/Qiskit-SG24/.env · QISKIT_IBM_TOKEN 설정됨


# 과제 7.1: 실험 결과 가져오기와 관리

이 노트북은 Qiskit Primitives job의 결과를 가져오고 관리하는 데 쓰이는 객체와 메서드를 실습으로 다룹니다.

**다루는 핵심 개념:**
- **목표 1:** `SamplerPubResult` 객체 이해하기
- **목표 2:** job 결과를 디스크에 저장하고 다시 불러오기
- **목표 3 & 4:** `RuntimeJob`과 `BasePrimitiveJob` 클래스의 속성과 메서드 살펴보기

모든 예제는 `AerSimulator`로 로컬에서 실행되도록 만들어져 있고, 클라우드 전용 부분은 따로 표시해 두었습니다.

정리하면:
```
job.result()          → PrimitiveResult   (PUB 여러 개의 리스트)
  result[0]           → SamplerPubResult  (PUB 하나)
    .data             → DataBin           (측정 데이터)
      .meas           → BitArray          (measure_all()의 기본 레지스터 이름이 'meas')
        .get_counts() → {'00': 530, '11': 494}
        .get_bitstrings()
    .metadata         → {'shots': 1024, ...}
```


## 설정: 간단한 Sampler job 실행

결과 객체를 살펴보려면 먼저 job을 하나 돌려야 합니다. 간단한 Bell circuit을 만들어 로컬 `BackendSamplerV2`로 실행합니다.

In [2]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.primitives import BackendSamplerV2 as Sampler
import numpy as np
import time
import json

# Bell state circuit을 만드는 helper 함수
def create_bell_circuit():
    """Bell state(최대로 얽힌 상태) circuit을 만든다."""
    qc = QuantumCircuit(2)
    qc.h(0)  # qubit 0에 Hadamard 적용
    qc.cx(0, 1)  # control=0, target=1인 CNOT 적용
    return qc

# 1. circuit과 backend 생성
circuit = create_bell_circuit()
circuit.measure_all()
local_backend = AerSimulator()

# 2. 로컬 Sampler를 만들고 job 실행
sampler = Sampler(backend=local_backend)
job = sampler.run([(circuit,)], shots=1024)

# 3. result 객체 받기
# 로컬 job은 보통 즉시 끝난다.
result = job.result()
print("로컬 Sampler job을 성공적으로 실행했습니다.")
print(f"Job ID: {job.job_id()}")

로컬 Sampler job을 성공적으로 실행했습니다.
Job ID: 9c268306-1e83-4377-b3b0-5e5f20100e8b


## 목표 1: `SamplerPubResult` 이해하기

V2 Primitives job의 결과를 받으면 `PrimitiveResult` 객체가 옵니다. 이건 `PubResult` 객체들의 리스트입니다 (제출한 circuit, 즉 "PUB" 하나당 하나). 방금 돌린 job 하나의 `PubResult`를 들여다봅시다.

### 속성: `data`와 `metadata`

`PubResult`에는 주요 속성이 두 개 있습니다:
*   `.data`: 핵심 결과 데이터가 담긴 객체 (Sampler면 bitstring, Estimator면 expectation value)
*   `.metadata`: job이 어떻게 실행됐는지에 대한 정보가 담긴 dictionary (예: shots 수)

In [3]:
# 첫 번째(이자 유일한) PUB의 결과 가져오기
pub_result = result[0]

# .data 속성 접근
# Sampler의 경우 'meas' 필드에 측정 결과가 담긴 DataBin 객체다.
bitstring_data = pub_result.data.meas

# .metadata 속성 접근
metadata = pub_result.metadata

print(f"--- Job '{job.job_id()}'의 PubResult ---")
print(f"\nMetadata: {metadata}")

# data 객체는 결과를 여러 형식으로 꺼내는 메서드를 갖고 있다
print(f"\n앞 5개 bitstring: {bitstring_data.get_bitstrings()[:5]}")
print(f"\nCounts dictionary: {bitstring_data.get_counts()}")

--- Job '9c268306-1e83-4377-b3b0-5e5f20100e8b'의 PubResult ---

Metadata: {'shots': 1024, 'circuit_metadata': {}}

앞 5개 bitstring: ['00', '11', '00', '11', '00']

Counts dictionary: {'00': 527, '11': 497}


### 메서드: `join_data()`

한 job에서 PUB 여러 개(circuit 여러 개)를 돌렸다면 `join_data()`로 결과를 합칠 수 있습니다.

In [4]:
# 서로 다른 circuit 두 개로 job 실행
circuit1 = create_bell_circuit()
circuit1.measure_all()

circuit2 = create_bell_circuit()
circuit2.x(0)  # X gate를 추가해 상태를 뒤집는다
circuit2.measure_all()

job_multi = sampler.run([(circuit1,), (circuit2,)], shots=512)
result_multi = job_multi.result()

print("첫 번째 circuit 결과:")
print(f"  Counts: {result_multi[0].data.meas.get_counts()}")

print("\n두 번째 circuit 결과:")
print(f"  Counts: {result_multi[1].data.meas.get_counts()}")

# 참고: join_data()는 구조가 같은 DataBin 객체들을 합칠 때 쓴다.
# 여기서는 각 PubResult와 그 data에 따로 접근하는 것으로 시연한다.

첫 번째 circuit 결과:
  Counts: {'11': 272, '00': 240}

두 번째 circuit 결과:
  Counts: {'01': 256, '10': 256}


## 목표 2: job 저장하고 다시 가져오기

실제 하드웨어에서 오래 걸리는 job은 노트북 안에서 기다리고 싶지 않을 겁니다. job을 제출하고 ID를 받아 둔 뒤 나중에 결과를 가져올 수 있습니다. 이 섹션에서는 결과를 디스크에 저장하고 불러오는 방법도 다룹니다.

### 결과를 디스크에 저장

`PrimitiveResult` 객체를 JSON 파일로 쉽게 저장할 수 있습니다. 결과를 공유하거나, job을 다시 돌리지 않고 후처리할 때 유용합니다.

In [5]:
# Qiskit 특유의 데이터 타입을 다루려면 Qiskit Runtime JSON encoder가 필요하다
from qiskit_ibm_runtime import RuntimeEncoder

# 'result'는 앞에서 job.result()로 받은 객체
with open("sampler_result.json", "w") as file:
    json.dump(result, file, cls=RuntimeEncoder)

print("Result 객체를 'sampler_result.json'에 저장했습니다.")

Result 객체를 'sampler_result.json'에 저장했습니다.


### 디스크에서 결과 불러오기

In [6]:
# 객체를 올바르게 복원하려면 Qiskit Runtime JSON decoder를 쓴다
from qiskit_ibm_runtime import RuntimeDecoder

with open("sampler_result.json", "r") as file:
    loaded_result = json.load(file, cls=RuntimeDecoder)

print("'sampler_result.json'에서 Result 객체를 불러왔습니다.")
print(f"\n불러온 Result 객체 타입: {type(loaded_result)}")

# 이제 전과 똑같이 데이터에 접근할 수 있다
print(f"\n불러온 데이터의 counts: {loaded_result[0].data.meas.get_counts()}")

'sampler_result.json'에서 Result 객체를 불러왔습니다.

불러온 Result 객체 타입: <class 'qiskit.primitives.containers.primitive_result.PrimitiveResult'>

불러온 데이터의 counts: {'00': 527, '11': 497}


### 클라우드 서비스에서 job 가져오기 (선택)

**참고:** 이 섹션은 `qiskit-ibm-runtime` 계정이 설정돼 있어야 합니다. 없으면 `AccountNotFoundError`가 나는데, 그게 정상입니다. `try...except`가 잡아서 노트북이 계속 진행됩니다.

In [7]:
import datetime

try:
    from qiskit_ibm_runtime import QiskitRuntimeService

    # 로컬에 계정이 저장돼 있을 때만 동작한다
    service = QiskitRuntimeService()

    three_months_ago = datetime.datetime.now() - datetime.timedelta(days=90)
    jobs_in_last_three_months = service.jobs(created_after=three_months_ago)

    print(f"최근 90일 job {len(list(jobs_in_last_three_months))}개를 찾았습니다.")
    if jobs_in_last_three_months:
        print("앞 3개:")
        for i, job in enumerate(jobs_in_last_three_months[:3]):
            print(f"  {i+1}. Job {job.job_id()} - Status: {job.status()}")

except Exception as e:
    print(f"클라우드 job 조회 건너뜀: {e}")
    print("IBM Quantum 계정이 설정돼 있지 않으면 정상입니다.")

최근 90일 job 1개를 찾았습니다.
앞 3개:
  1. Job d9qqmonpemts73crmhbg - Status: CANCELLED


### 특정 job을 ID로 가져오기

job ID를 알면 바로 가져올 수 있습니다. job을 제출하고 ID를 저장해 둔 뒤 나중에 돌아와 결과를 받을 때 유용합니다.

In [8]:
try:
    from qiskit_ibm_runtime import QiskitRuntimeService

    service = QiskitRuntimeService()

    # 시연용으로 가장 최근에 성공한 job을 찾는다
    successful_jobs = [j for j in service.jobs(limit=100) if j.status() == "DONE"]

    if successful_jobs:
        successful_job = successful_jobs[0]
        job_id = successful_job.job_id()
        print(f"성공한 job 찾음: {job_id}")

        # ID로 job 가져오기
        retrieved_job = service.job(job_id)
        retrieved_result = retrieved_job.result()

        print(f"Job {job_id}을(를) 성공적으로 가져왔습니다")
        print(f"Result: {retrieved_result}")
    else:
        print("최근 기록에 성공한 job이 없습니다.")

except Exception as e:
    print(f"job 조회 건너뜀: {e}")
    print("IBM Quantum 계정이 설정돼 있지 않으면 정상입니다.")

성공한 job 찾음: d6tp2maf84ks73ddff4g
Job d6tp2maf84ks73ddff4g을(를) 성공적으로 가져왔습니다
Result: PrimitiveResult([SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=100, num_bits=1>)), metadata={'circuit_metadata': {}})], metadata={'execution': {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2026-03-19 06:02:05', stop='2026-03-19 06:02:05', size=100>)])}, 'version': 2})


### 참고! 여러개 받아오기: `service.jobs`를 이용하자.

```
service.jobs(limit=10, skip=0, backend_name=None, pending=None, program_id=None,
             instance=None, job_tags=None, session_id=None,
             created_after=None, created_before=None, descending=True)
```

## 목표 3 & 4: `RuntimeJob`과 `BasePrimitiveJob`

job을 실행하면 `Job` 객체를 돌려받습니다. 로컬 `BackendPrimitives`면 `PrimitiveJob`, runtime 서비스면 `RuntimeJob`입니다. 둘 다 `BasePrimitiveJob`을 상속하며, job 상태를 관리하고 조회하는 공통 메서드를 공유합니다.

**중요:** 일부 메서드는 `RuntimeJob`(클라우드 job)에만 있고 `PrimitiveJob`(로컬 job)에는 없습니다. 어느 쪽인지 명확히 표시하겠습니다.

### job 관리 예제

로컬 시뮬레이터에서 job을 돌리고, job 객체의 메서드로 상태를 모니터링해 봅시다. 로컬 job은 아주 빨리 끝나지만, 실제 하드웨어에서는 `QUEUED`와 `RUNNING` 상태에 한동안 머물 수 있습니다.

In [9]:
from qiskit.circuit.library import EfficientSU2
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.providers.jobstatus import JobStatus

# 시뮬레이터가 잠깐이라도 일하도록 조금 더 복잡한 circuit을 만든다
circuit = EfficientSU2(10, reps=4, entanglement='linear')
circuit.measure_all()
params = np.random.rand(circuit.num_parameters)

# circuit transpile
pm = generate_preset_pass_manager(optimization_level=1, backend=local_backend)
isa_circuit = pm.run(circuit)

# job 제출
job = sampler.run([(isa_circuit, params)])

print(f"제출한 job ID: {job.job_id()}")

# --- job 상태 polling ---
# 오래 걸리는 클라우드 job을 확인하는 방식을 흉내 낸 루프다.
max_checks = 50
checks = 0
while not job.in_final_state() and checks < max_checks:
    # JobStatus는 QUEUED, RUNNING, DONE, ERROR, CANCELLED 같은 값을 갖는 Enum이다
    status = job.status()
    print(f"현재 job 상태: {status}")
    time.sleep(0.1)  # 다시 확인하기 전에 잠깐 대기
    checks += 1

final_status = job.status()
print(f"\njob 최종 상태: {final_status}")

# --- 결과 접근 ---
if job.done():
    result = job.result()
    print("job이 성공적으로 완료됐습니다!")
    print(f"Sample counts: {result[0].data.meas.get_counts()}")
else:
    # 로컬 job은 result()를 호출할 때 에러가 바로 raise된다
    # 클라우드 job에는 errored(), error_message() 같은 메서드가 추가로 있다
    status = job.status()
    print(f"job 상태: {status}")

/tmp/ipykernel_184055/99860533.py:6: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  circuit = EfficientSU2(10, reps=4, entanglement='linear')


제출한 job ID: 60854db5-de42-4bd8-a1f8-36bb79df5024
현재 job 상태: JobStatus.RUNNING

job 최종 상태: JobStatus.DONE
job이 성공적으로 완료됐습니다!
Sample counts: {'1101000000': 11, '0100010001': 27, '1011010000': 4, '0100011011': 1, '0010010001': 10, '1100011101': 1, '1101100010': 1, '1011101110': 9, '0010111000': 2, '1100111101': 2, '0111110101': 1, '0111100000': 7, '0100100101': 1, '0010101000': 2, '1100010011': 2, '1000001100': 6, '1001101101': 5, '1000010101': 8, '0010101110': 2, '1001110000': 2, '0111000101': 4, '0010101010': 1, '0010001100': 6, '0011111001': 2, '0110111001': 6, '0111001001': 3, '0001000001': 8, '0111001110': 2, '1000110000': 3, '1101011100': 1, '0111100101': 1, '1101111000': 1, '0111110000': 7, '0110110011': 1, '1101000101': 6, '1101100111': 1, '1001110011': 2, '1100001100': 1, '1100010000': 4, '1100101100': 1, '1110110111': 2, '0010001001': 6, '1010011001': 1, '0000001111': 1, '1110101100': 1, '0001000110': 2, '1001010101': 11, '1011101111': 12, '1001101000': 11, '1001010001': 10, '00

### 주요 job 속성과 메서드

`job` 객체에는 유용한 메서드와 속성이 많습니다. 가장 중요한 것들:

**PrimitiveJob(로컬)과 RuntimeJob(클라우드) 둘 다에 있는 메서드:**
* `job.status()`: job 상태 반환 (예: `JobStatus.RUNNING`)
* `job.in_final_state()`: job이 done, errored, cancelled 중 하나면 `True`
* `job.done()`: job이 **성공적으로** 끝났을 때만 `True`
* `job.running()`: 현재 실행 중이면 `True`
* `job.cancelled()`: 취소됐으면 `True`
* `job.result()`: job이 최종 상태가 될 때까지 **실행을 막고(block)** 기다렸다가 `PrimitiveResult` 반환
* `job.job_id()`: job의 고유 문자열 ID 반환
* `job.cancel()`: 대기 중이거나 실행 중인 job 취소 시도

**RuntimeJob(클라우드 job)에만 있는 메서드:**
* `job.errored()`: job이 실패했으면 `True` (로컬 PrimitiveJob에는 없음)
* `job.error_message()`: job이 실패했을 때 에러 메시지 반환
* `job.backend()`: backend 객체 반환 (모든 job 타입에서 되는 건 아님)
* `job.queue_info()`: 대기열 위치 정보
* `job.queue_position()`: 대기열에서의 위치
* `job.logs()`: job 로그 조회
* `job.metrics()`: job 성능 지표
* `job.update_tags()`: job 태그 갱신

* 추가! job.wait_for_final_state()

**RuntimeJob 전용 속성:**
* `job.creation_date`
* `job.tags`
* `job.session_id`
* `job.usage_estimation`

### 추가: Jobstatus 종류
```
QUEUED     'job is queued'
RUNNING    'job is actively running'
CANCELLED  'job has been cancelled'
DONE       'job has successfully run'   ← 
ERROR      'job incurred error'
```

In [10]:
# --- 모든 job 타입에서 동작하는 메서드 시연 ---

# job ID
print(f"Job ID: {job.job_id()}")

# 최종 상태인지 확인 (지금은 True여야 함)
print(f"최종 상태인가? {job.in_final_state()}")

# 개별 상태 플래그 확인 (모든 job에서 동작)
print(f"done? {job.done()}")
print(f"running? {job.running()}")
print(f"cancelled? {job.cancelled()}")

# 참고: errored()는 RuntimeJob에만 있다
# 로컬 job은 status()를 직접 확인한다:
print(f"Job status: {job.status()}")

Job ID: 60854db5-de42-4bd8-a1f8-36bb79df5024
최종 상태인가? True
done? True
running? False
cancelled? False
Job status: JobStatus.DONE


## 요약

이 노트북에서 다룬 것:

1. **PubResult 구조**: 결과가 담긴 `data`와 `metadata` 속성
2. **결과 저장/불러오기**: `RuntimeEncoder`와 `RuntimeDecoder`로 결과를 디스크에 보존
3. **job 조회**: IBM Quantum에서 job 가져오기 (클라우드 전용)
4. **job 관리**: PrimitiveJob(로컬)과 RuntimeJob(클라우드)의 차이
   - 둘 다에서 동작하는 공통 메서드
   - 모니터링과 에러 처리를 위한 클라우드 전용 메서드

이 도구들은 오래 걸리는 양자 실험을 관리하는 데 필수적입니다. 특히 job이 오랫동안 대기열에 머물 수 있는 실제 하드웨어에서요.

## 연습문제

**1. V2 `PubResult` 객체에서 핵심 데이터(bitstring이나 expectation value 등)는 어디에 저장되나?**

A) `.metadata` 속성

B) `.data` 속성

C) `.results` 속성

D) `.value` 속성

***정답:***
<Details>
<br/>

**B) `.data` 속성**

`SamplerPubResult`의 public 속성은 딱 3개: `data`, `metadata`, `join_data`.

```
result[0].data.meas.get_counts()   → {'00': 530, '11': 494}   ← 데이터
result[0].metadata                 → {'shots': 1024, ...}     ← 부가정보
```

**오답 해설**

| 보기 | 왜 틀림 |
| --- | --- |
| A) `.metadata` | 존재하지만 **shots 수, precision 같은 실행 정보**가 들어감. bitstring·expectation value는 없음 |
| C) `.results` | **존재하지 않음**. V1 시절 `Result` 객체의 `.results`를 떠올리게 하는 함정 |
| D) `.value` | **존재하지 않음**. Estimator의 `.data.evs`처럼 값은 `.data` 아래 필드로 들어감 |

Sampler는 `.data.meas`(classical register 이름), Estimator는 `.data.evs` / `.data.stds`. 둘 다 `.data` 밑이다.
</Details>

---

**2. job이 성공적으로 완료됐는지 확실하게 확인하려면 어떤 메서드를 써야 하나?**

A) `job.in_final_state()`

B) `job.running()`

C) `job.done()`

D) `job.status() == 'COMPLETED'`

***정답:***
<Details>
<br/>

**C) `job.done()`** — 상태가 `DONE`일 때만 `True`. 취소·에러는 제외.

실측: 완료된 로컬 job에서 `done=True, running=False, cancelled=False, in_final_state=True`.

**오답 해설**

| 보기 | 왜 틀림 |
| --- | --- |
| A) `in_final_state()` | **가장 헷갈리는 오답.** `DONE`, `ERROR`, `CANCELLED` 셋 다 `True`. "끝났다"지 "성공했다"가 아님. 이 노트북 [23]의 polling 루프 `while not job.in_final_state()`가 이거 — *기다리는 용도*로는 맞지만 성공 판정은 아님 |
| B) `running()` | 실행 *중*인지. 끝났으면 `False`라 반대 의미 |
| D) `status() == 'COMPLETED'` | 두 가지로 틀림. **① `COMPLETED`라는 상태는 없음** — `JobStatus` 멤버는 `INITIALIZING, QUEUED, VALIDATING, RUNNING, CANCELLED, DONE, ERROR`. **② 로컬 `PrimitiveJob.status()`는 문자열이 아니라 `JobStatus` enum 반환** — `'COMPLETED'`는커녕 `'DONE'`과 비교해도 `False`. (클라우드 `RuntimeJobV2.status()`는 문자열 `"DONE"` 반환이라 타입이 다름 — 이것도 시험 포인트) |

이 노트북 [23]이 정석 흐름: `in_final_state()`로 기다리고 → `done()`으로 성공 확인 → `result()`.
</Details>

---

**3. 다음 중 클라우드 `RuntimeJob`에는 있지만 로컬 `PrimitiveJob`에는 보통 없는 메서드는?**

A) `job.status()`

B) `job.job_id()`

C) `job.error_message()`

D) `job.result()`

***정답:***
<Details>
<br/>

**C) `job.error_message()`**

실측한 차이 (`dir()` 비교):

```
공통 (둘 다 있음)   : cancel, cancelled, done, in_final_state, job_id, result, running, status
RuntimeJobV2 전용   : error_message, errored, wait_for_final_state, backend, session_id,
                      usage, metrics, logs, creation_date, tags, ...
PrimitiveJob 전용   : (없음)
```

**오답 해설**

| 보기 | 왜 틀림 |
| --- | --- |
| A) `status()` | 공통 |
| B) `job_id()` | 공통. 로컬도 UUID를 준다 (위 출력의 `Job ID:` 참고) |
| D) `result()` | 공통 |

왜 로컬엔 `error_message()`가 없나 — 로컬 job은 에러가 나면 **`result()` 호출 시 즉시 예외**로 터진다. 나중에 메시지를 따로 조회할 필요가 없다. 클라우드는 비동기라 실패한 job의 원인을 나중에 꺼내봐야 하므로 `errored()` + `error_message()`가 따로 있다. [23]의 `else` 블록 주석이 정확히 이 얘기.

`wait_for_final_state()`도 클라우드 전용인데, 로컬은 `result()`가 이미 blocking이라 필요 없어서다.
</Details>